<a href="https://colab.research.google.com/github/oshaajayaweera/Databases-and-Analytics/blob/main/01_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

library(sqldf)
library(dplyr)
library(ggplot2)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
base_url <- "https://raw.githubusercontent.com/oshaajayaweera/Databases-and-Analytics/main/"

customers <- read.csv(paste0(base_url, "customers.csv"))
orders <- read.csv(paste0(base_url, "orders.csv"))
deliveries <- read.csv(paste0(base_url, "deliveries.csv"))
drivers <- read.csv(paste0(base_url, "drivers.csv"))
vehicles <- read.csv(paste0(base_url, "vehicles.csv"))
hubs <- read.csv(paste0(base_url, "hubs.csv"))
complaints <- read.csv(paste0(base_url, "complaints.csv"))
incidents <- read.csv(paste0(base_url, "incidents.csv"))
app_events <- read.csv(paste0(base_url, "app_events.csv"))
data_dictionary <- read.csv(paste0(base_url, "data_dictionary.csv"))

In [3]:
head(deliveries)
str(deliveries)
head(orders)
head(hubs)

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58


'data.frame':	950 obs. of  13 variables:
 $ delivery_id                  : chr  "DL00001" "DL00002" "DL00003" "DL00004" ...
 $ order_id                     : chr  "O00938" "O00004" "O00639" "O00313" ...
 $ driver_id                    : chr  "D004" "D138" "D006" "D116" ...
 $ vehicle_id                   : chr  "V056" "V007" "V049" "V055" ...
 $ hub_id                       : chr  "H05" "H02" "H02" "H02" ...
 $ dispatch_time                : chr  "2024-06-18 10:57:00" "2025-01-11 18:45:00" "2025-06-02 20:39:00" "2024-03-08 23:31:00" ...
 $ delivery_completed_at        : chr  "2024-06-19 09:05:59.904311" "2025-01-11 17:39:00.000000" "2025-06-02 21:45:32.366770" "2024-03-09 23:30:08.103702" ...
 $ delivery_status              : chr  "Failed" "OnTime" "OnTime" "Delayed" ...
 $ route_distance_km            : num  17.26 10.34 7.92 16.42 14.52 ...
 $ manual_route_override_count  : int  1 1 0 0 1 0 0 1 1 1 ...
 $ proof_of_completion_missing  : int  0 0 0 0 0 0 0 0 0 0 ...
 $ customer_rating_p

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


,hub_id,hub_name,zone,hub_type,capacity_score
,<chr>,<chr>,<chr>,<chr>,<int>
1,H01,North Exchange,North,Dispatch,82
2,H02,South Link,South,Dispatch,78
3,H03,East Dock,East,Warehouse,74
4,H04,West Gate,West,Dispatch,69
5,H05,Central Core,Central,Control,88
6,H06,Airport Hub,Airport,Dispatch,71


In [4]:
query1 <- sqldf("
SELECT
    delivery_id,
    order_id,
    driver_id,
    hub_id,
    dispatch_time,
    delivery_completed_at,
    delivery_status,
    ROUND((julianday(delivery_completed_at) - julianday(dispatch_time)) * 1440, 2) AS delivery_minutes
FROM deliveries
WHERE delivery_completed_at IS NOT NULL
  AND dispatch_time IS NOT NULL
  AND ROUND((julianday(delivery_completed_at) - julianday(dispatch_time)) * 1440, 2) > 60
ORDER BY delivery_minutes DESC
")

query1

delivery_id,order_id,driver_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,delivery_minutes
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
DL00386,O00135,D144,H05,2025-04-25 03:12:00,2025-04-26 22:39:24.905513,Failed,2607.42
DL00387,O00274,D016,H01,2025-10-03 21:33:00,2025-10-05 15:40:35.899437,Failed,2527.60
DL00033,O00885,D041,H06,2024-11-17 14:01:00,2024-11-19 06:51:25.201294,Failed,2450.42
DL00530,O00756,D095,H01,2025-02-03 18:06:00,2025-02-05 09:39:56.840261,Failed,2373.95
DL00026,O00906,D092,H04,2025-02-04 11:16:00,2025-02-06 01:48:45.831712,Failed,2312.76
DL00806,O00128,D117,H07,2025-12-28 14:40:00,2025-12-30 04:18:53.684717,Delayed,2258.89
DL00497,O00192,D083,H08,2025-08-18 14:17:00,2025-08-20 03:34:36.170242,Failed,2237.60
DL00472,O00042,D113,H01,2025-12-01 19:10:00,2025-12-03 08:16:03.110618,Delayed,2226.05
DL00775,O00153,D109,H05,2025-02-09 09:08:00,2025-02-10 21:51:56.051970,Delayed,2203.93


In [5]:
query2 <- sqldf("
SELECT
    delivery_id,
    order_id,
    driver_id,
    hub_id,
    vehicle_id,
    delivery_status
FROM deliveries
WHERE delivery_status = 'Failed'
")

query2

delivery_id,order_id,driver_id,hub_id,vehicle_id,delivery_status
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DL00001,O00938,D004,H05,V056,Failed
DL00010,O00836,D058,H08,V057,Failed
DL00012,O01207,D051,H05,V017,Failed
DL00022,O01027,D088,H07,V011,Failed
DL00026,O00906,D092,H04,V055,Failed
DL00033,O00885,D041,H06,V075,Failed
DL00038,O00727,D096,H02,V002,Failed
DL00039,O00542,D090,H08,V034,Failed
DL00040,O00919,D027,H04,V036,Failed


In [6]:
query3 <- sqldf("
SELECT
    hub_id,
    COUNT(delivery_id) AS total_deliveries,
    SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM deliveries
GROUP BY hub_id
ORDER BY failed_deliveries DESC, delayed_deliveries DESC
")

query3

hub_id,total_deliveries,delayed_deliveries,failed_deliveries
<chr>,<int>,<int>,<int>
H08,128,22,26
H05,115,25,23
H01,136,26,17
H04,127,28,16
H06,104,27,15
H07,115,25,14
H03,119,23,11
H02,106,26,10


In [7]:
query4 <- sqldf("
SELECT
    hub_id,
    ROUND(AVG(manual_route_override_count), 2) AS avg_route_overrides,
    ROUND(AVG(route_distance_km), 2) AS avg_route_distance
FROM deliveries
GROUP BY hub_id
ORDER BY avg_route_overrides DESC
")

query4

hub_id,avg_route_overrides,avg_route_distance
<chr>,<dbl>,<dbl>
H08,1.11,12.82
H07,1.05,14.29
H01,1.03,13.64
H05,0.95,14.32
H02,0.92,14.17
H06,0.91,14.41
H03,0.89,14.52
H04,0.87,13.38


In [8]:
query4 <- sqldf("
SELECT
    hub_id,
    ROUND(AVG(manual_route_override_count), 2) AS avg_route_overrides,
    ROUND(AVG(route_distance_km), 2) AS avg_route_distance
FROM deliveries
GROUP BY hub_id
ORDER BY avg_route_overrides DESC
")

query4

hub_id,avg_route_overrides,avg_route_distance
<chr>,<dbl>,<dbl>
H08,1.11,12.82
H07,1.05,14.29
H01,1.03,13.64
H05,0.95,14.32
H02,0.92,14.17
H06,0.91,14.41
H03,0.89,14.52
H04,0.87,13.38


In [9]:
query5 <- sqldf("
SELECT
    delivery_status,
    ROUND(AVG(fuel_or_charge_cost), 2) AS avg_fuel_charge_cost
FROM deliveries
GROUP BY delivery_status
ORDER BY avg_fuel_charge_cost DESC
")

query5

delivery_status,avg_fuel_charge_cost
<chr>,<dbl>
Failed,13.15
Delayed,13.14
OnTime,12.68
